# 07 · Account Reachout Recommendations
AI-powered prioritized list of accounts the specialist team should reach out to.
Signals used: days since last contact, use case stage, Gong conversation signals, and pipeline health.

In [ ]:
import datetime, json
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, matplotlib.gridspec as gs
import pandas as pd
from IPython.display import display, HTML
plt.switch_backend("agg")

SF_BLUE="#29B5E8"; SF_TEAL="#00A9CE"; SF_GREEN="#36B37E"
SF_ORANGE="#FF8B00"; SF_DARK="#0A2859"; SF_GRAY="#D0D0D0"; BG="#F9FAFB"

def clean(ax):
    ax.set_facecolor(BG)
    for s in ax.spines.values(): s.set_visible(False)

def fmt_eacv(v):
    if not v or (isinstance(v,float) and str(v)=="nan"): return "$0"
    v=float(v)
    return f"${v/1e6:.1f}M" if v>=1e6 else f"${v/1e3:.0f}K" if v>=1e3 else f"${v:.0f}"

def html_table(df, row_color_fn=None):
    if hasattr(df, 'to_pandas'): df = df.to_pandas()
    html = "<table style='border-collapse:collapse;width:100%;font-size:13px;font-family:sans-serif'>"
    html += "<tr style='background:#0A2859;color:white'>" + "".join("<th style='padding:8px 12px;text-align:left'>" + str(col) + "</th>" for col in df.columns) + "</tr>"
    for i, (_, r) in enumerate(df.iterrows()):
        bg = row_color_fn(r) if row_color_fn else ("#f8f9fa" if i%2==0 else "white")
        html += "<tr style='background:" + bg + "'>" + "".join("<td style='padding:7px 12px'>" + str(v if v is not None else '') + "</td>" for v in r) + "</tr>"
    html += "</table>"
    display(HTML(html))

TEAM = {
    "Jim Lebonitte": "005VI00000Z0y6bYAB",
    "Michael Hamilton": "005VI00000ExC3VYAV",
    "Patrick Sheehan": "0053r00000BblkZAAR",
    "Phani Alapaty": "005VI00000HajknYAB",
    "Steve Mitchener": "0050Z000009IrKRQA0",
    "Zaki Bajwa": "005VI00000XibQfYAJ"
}
TEAM_IDS   = list(TEAM.values())
team_ids_str = "', '".join(TEAM_IDS)
ACT_TABLE  = "TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES"

fy_start   = datetime.date(2026, 2, 1)
q2_start   = datetime.date(2026, 5, 1)
today      = datetime.date.today()
fy_start_str = str(fy_start)
q2_start_str = str(q2_start)
today_str    = str(today)
print("Setup complete ✓")

# ─── FILTERS ─────────────────────────────────────────────
cold_threshold_days   = 21   # flag accounts with no contact in >N days
uc_stages_to_watch    = [2, 3]  # Scoping and Technical Validation
# ─────────────────────────────────────────────────────────
print(f"Reachout recommendations as of {today_str}")
print(f"Cold threshold: {cold_threshold_days} days")


## Priority 1 — Accounts with Active Use Cases Going Cold

In [ ]:
%%sql -r cold_with_uc
SELECT
    a.SF_ACCOUNT_ID,
    COALESCE(a.CUSTOMER, d.ACCOUNT_NAME)    AS account,
    a.MEETING_SE_NAME                        AS last_specialist,
    MAX(a.MEETING_DATE)                      AS last_contact,
    DATEDIFF('day',MAX(a.MEETING_DATE),CURRENT_DATE()) AS days_cold,
    d.USE_CASE_NAME,
    d.USE_CASE_STAGE,
    d.STAGE_NUMBER,
    d.DECISION_DATE,
    d.USE_CASE_LEAD_SE_NAME                  AS uc_lead_se,
    ROUND(d.USE_CASE_EACV/1e3,0)            AS eacv_k,
    d.USE_CASE_RISK_LEVEL
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES a
JOIN MDM.MDM_INTERFACES.DIM_USE_CASE d ON a.SF_ACCOUNT_ID = d.ACCOUNT_ID
WHERE d.IS_LOST = FALSE
  AND d.STAGE_NUMBER BETWEEN 1 AND 6
GROUP BY 1,2,3,6,7,8,9,10,11,12
  -- HAVING: last contact > threshold OR account in risky stage
HAVING days_cold >= {{cold_threshold_days}}
   OR (d.STAGE_NUMBER IN (2,3) AND days_cold >= 14)
ORDER BY d.STAGE_NUMBER DESC, days_cold DESC
LIMIT 30

In [ ]:
cold_with_uc = cold_with_uc.to_pandas() if hasattr(cold_with_uc, "to_pandas") else cold_with_uc
def uc_row_color(r):
    days = int(r.get("DAYS_COLD",0) or 0)
    stage = int(r.get("STAGE_NUMBER",0) or 0)
    risk = str(r.get("USE_CASE_RISK_LEVEL","") or "")
    if "High" in risk or days > 45: return "#fdecea"
    if stage >= 3 and days > 21: return "#fff3cd"
    return "#f0f8ff"
html_table(cold_with_uc, row_color_fn=uc_row_color)
print("Red=High risk / >45 days cold  |  Yellow=Stage 3+ and >21 days  |  Blue=Active")

## Priority 2 — Accounts with Meetings but No Use Case (Opportunity to Create)

In [ ]:
%%sql -r no_uc_accounts
SELECT
    a.SF_ACCOUNT_ID,
    COALESCE(a.CUSTOMER,'(unknown)')    AS account,
    COUNT(*)                             AS total_meetings,
    MAX(a.MEETING_DATE)                  AS last_contact,
    DATEDIFF('day',MAX(a.MEETING_DATE),CURRENT_DATE()) AS days_since,
    a.MEETING_SE_NAME                    AS specialist,
    a.AE_NAME,
    a.AE_EMAIL,
    MAX(a.OPPORTUNITY_NAME)              AS opportunity,
    MAX(a.OPPORTUNITY_ID)                AS opportunity_id
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES a
WHERE a.USE_CASE_TAGGED_IN_SF = 'No'
  AND a.SF_ACCOUNT_ID IS NOT NULL
GROUP BY 1,2,6,7,8
ORDER BY total_meetings DESC, days_since ASC
LIMIT 25

In [ ]:
no_uc_accounts = no_uc_accounts.to_pandas() if hasattr(no_uc_accounts, "to_pandas") else no_uc_accounts
print(f"Accounts with meetings but no use case: {len(no_uc_accounts)}")
html_table(no_uc_accounts)

## Priority 3 — Gong Signal: Accounts Showing High Intent

In [ ]:
%%sql -r high_intent
SELECT
    CUSTOMER,
    SF_ACCOUNT_ID,
    MEETING_SE_NAME         AS specialist,
    MEETING_DATE,
    MEETING_TITLE,
    LEFT(SUMMARY, 400)      AS summary_preview,
    LEFT(NEXT_STEPS, 300)   AS next_steps_preview,
    USE_CASE_TAGGED_IN_SF   AS uc_status
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE SUMMARY IS NOT NULL
  AND MEETING_DATE >= DATEADD('day', -30, CURRENT_DATE())
  AND (
    LOWER(SUMMARY) LIKE '%poc%'
    OR LOWER(SUMMARY) LIKE '%proof of concept%'
    OR LOWER(SUMMARY) LIKE '%proposal%'
    OR LOWER(SUMMARY) LIKE '%migrate%'
    OR LOWER(SUMMARY) LIKE '%committed%'
    OR LOWER(SUMMARY) LIKE '%move forward%'
    OR LOWER(SUMMARY) LIKE '%budget%'
    OR LOWER(SUMMARY) LIKE '%next steps%'
    OR LOWER(NEXT_STEPS) LIKE '%follow up%'
    OR LOWER(NEXT_STEPS) LIKE '%schedule%'
  )
ORDER BY MEETING_DATE DESC LIMIT 20

In [ ]:
high_intent = high_intent.to_pandas() if hasattr(high_intent, "to_pandas") else high_intent
html_table(high_intent)

## AI — Personalized Reachout Recommendations

In [ ]:
%%sql -r reachout_context
SELECT
    a.SF_ACCOUNT_ID,
    COALESCE(a.CUSTOMER, d.ACCOUNT_NAME) AS account,
    a.MEETING_SE_NAME                     AS specialist,
    MAX(a.MEETING_DATE)                   AS last_contact,
    DATEDIFF('day',MAX(a.MEETING_DATE),CURRENT_DATE()) AS days_cold,
    d.USE_CASE_NAME,
    d.USE_CASE_STAGE,
    d.USE_CASE_RISK_LEVEL,
    d.DECISION_DATE,
    LISTAGG(DISTINCT LEFT(a.SUMMARY,300), ' | ') AS recent_summaries
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES a
LEFT JOIN MDM.MDM_INTERFACES.DIM_USE_CASE d ON a.SF_ACCOUNT_ID = d.ACCOUNT_ID
WHERE a.SF_ACCOUNT_ID IS NOT NULL
  AND (d.IS_LOST = FALSE OR d.IS_LOST IS NULL)
  AND DATEDIFF('day', a.MEETING_DATE, CURRENT_DATE()) BETWEEN 14 AND 60
GROUP BY 1,2,3,6,7,8,9
HAVING MAX(a.SUMMARY) IS NOT NULL
ORDER BY days_cold DESC LIMIT 15

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
reachout_context = reachout_context.to_pandas() if hasattr(reachout_context, "to_pandas") else reachout_context
accounts_text = ""
for _, r in reachout_context.iterrows():
    accounts_text += (
        f"Account: {r['ACCOUNT']} | Specialist: {r['SPECIALIST']} | "
        f"Days cold: {r['DAYS_COLD']} | Stage: {r['USE_CASE_STAGE'] or 'No UC'} | "
        f"Decision date: {r['DECISION_DATE'] or 'Unknown'} | "
        f"Risk: {r['USE_CASE_RISK_LEVEL'] or 'Unknown'}\n"
        f"Recent conversation: {r['RECENT_SUMMARIES'][:400]}\n\n"
    )

if accounts_text.strip():
    prompt = (
        f"You are advising a Snowflake specialist team on which accounts to prioritize for outreach (as of {today_str}).\n"
        "For each account below, provide:\n"
        "1. PRIORITY (High/Medium/Low) and why\n"
        "2. RECOMMENDED ACTION in one sentence (e.g. 'Schedule architecture review', 'Follow up on POC results')\n"
        "3. KEY TALKING POINT based on last conversation\n\n"
        "Accounts:\n" + accounts_text[:6000]
    )
    result = session.sql("SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-7b', ?)", [prompt]).collect()[0][0]
    print(f"=== AI Reachout Recommendations ({today_str}) ===\n")
    print(result)
else:
    print("Not enough context to generate recommendations — ensure Gong summaries are populated.")

## Summary Reachout Action Table

In [ ]:
%%sql -r action_table
SELECT
    COALESCE(a.CUSTOMER, d.ACCOUNT_NAME)    AS account,
    a.SF_ACCOUNT_ID,
    a.MEETING_SE_NAME                        AS specialist,
    a.AE_NAME,
    a.AE_EMAIL,
    MAX(a.MEETING_DATE)                      AS last_contact,
    DATEDIFF('day',MAX(a.MEETING_DATE),CURRENT_DATE()) AS days_cold,
    d.USE_CASE_STAGE,
    d.DECISION_DATE,
    CASE
        WHEN d.STAGE_NUMBER >= 3 AND DATEDIFF('day',MAX(a.MEETING_DATE),CURRENT_DATE()) > 14 THEN 'HIGH'
        WHEN d.STAGE_NUMBER BETWEEN 1 AND 2 AND DATEDIFF('day',MAX(a.MEETING_DATE),CURRENT_DATE()) > 21 THEN 'MEDIUM'
        WHEN d.STAGE_NUMBER IS NULL AND DATEDIFF('day',MAX(a.MEETING_DATE),CURRENT_DATE()) < 30 THEN 'LOW - Create UC'
        ELSE 'REVIEW'
    END AS priority
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES a
LEFT JOIN MDM.MDM_INTERFACES.DIM_USE_CASE d ON a.SF_ACCOUNT_ID = d.ACCOUNT_ID
WHERE a.SF_ACCOUNT_ID IS NOT NULL
  AND (d.IS_LOST = FALSE OR d.IS_LOST IS NULL)
GROUP BY 1,2,3,4,5,8,9, d.STAGE_NUMBER
ORDER BY priority, days_cold DESC LIMIT 40

In [ ]:
action_table = action_table.to_pandas() if hasattr(action_table, "to_pandas") else action_table
def priority_color(r):
    p = str(r.get("PRIORITY",""))
    if "HIGH" in p: return "#fdecea"
    if "MEDIUM" in p: return "#fff3cd"
    if "Create UC" in p: return "#d4edda"
    return "white"
html_table(action_table, row_color_fn=priority_color)
print("Red=High priority reachout  |  Yellow=Medium  |  Green=Low/Create UC")